# SVD-compressed SwiGLU MLP on a **free** Colab TPU

Self-contained NeuronMM / SVD-Flash kernel (paper *NeuronMLP*, arXiv:2510.25977),
auto-tuned for whatever TPU Colab gives you — **v5e-1** by default (not the v4 the
repo's `svd_mlp_tpu.py` hardcodes), sometimes **v2-8**.

**Steps:** `Runtime → Change runtime type → TPU → Save`, then `Runtime → Run all`.

Cell 1 *compiles a tiny Pallas kernel as a probe*. Colab images sometimes ship a
jaxlib newer than their libtpu (a Mosaic `Unsupported version` error); if so the
cell installs a matched pair and asks you to **Restart session** and Run all once more.


### 1. Probe Pallas; install a matched jax[tpu]+libtpu pair only if it doesn't compile


In [ ]:
import subprocess, sys

def _pallas_works():
    """True only if a real Pallas/Mosaic kernel compiles on the TPU."""
    try:
        import jax, jax.numpy as jnp
        from jax.experimental import pallas as pl
        if jax.default_backend() != "tpu":
            return False, "no TPU backend"
        def _k(x_ref, o_ref):
            o_ref[...] = x_ref[...] + 1.0
        f = pl.pallas_call(_k, out_shape=jax.ShapeDtypeStruct((8, 128), jnp.float32))
        jax.block_until_ready(f(jnp.zeros((8, 128), jnp.float32)))
        return True, "pallas compiles"
    except Exception as e:
        return False, f"{type(e).__name__}: {str(e)[:140]}"

ok, why = _pallas_works()
print("probe:", why)
if not ok:
    print(">> installing a matched jax[tpu] + libtpu pair ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "jax[tpu]",
                    "-f", "https://storage.googleapis.com/jax-releases/libtpu_releases.html"],
                   check=True)
    print("\n>> DONE. Now do: Runtime -> Restart session, then Runtime -> Run all again.")
    raise SystemExit("Restart the session, then re-run (this is expected, not an error).")
print(">> jax + libtpu + Pallas all good.")


### 2. Confirm the backend and see which TPU you got


In [ ]:
import jax
print('jax', jax.__version__)
print('devices:', jax.devices())
print('device_kind:', jax.devices()[0].device_kind)
assert jax.default_backend() == 'tpu', \
    'Not on TPU - Runtime -> Change runtime type -> TPU, then Restart session.'


### 3. The implementation
Kernels + auto TPU-spec detection + VMEM autotune + reference + correctness/benchmark.


In [ ]:
"""
colab_svd_mlp.py — self-contained, Colab-ready SVD-compressed SwiGLU MLP on TPU.

A single-file port of NeuronMM/SVD-Flash (Song et al., arXiv:2510.25977,
Algorithm 2/3) tuned to run on a **free Colab/Kaggle TPU** — which is *not* a v4.
Colab currently hands out a single-chip **TPU v5e-1** (sometimes a **v2-8**);
Kaggle gives a **v3-8**. The existing `svd_mlp_tpu.py` hardcodes the v4 spec, so
the block sizes are wrong on those cores. This file fixes that with three things:

  1. AUTO-SPEC      — reads `jax.devices()[0].device_kind` and builds the
                      analytical block-size model's spec for whatever core you got.
  2. AUTO-FALLBACK  — tries the fast fused path first; if Mosaic overflows VMEM it
                      automatically streams the weights (K-blocking) and shrinks the
                      tile until it compiles. You never have to hand-pick `BK`.
  3. ONE FILE       — kernels + block model + correctness + benchmark, no local
                      imports. Upload just this file, or paste it into one cell.

Run (identical command on a laptop or a TPU; it auto-detects the backend):

    python colab_svd_mlp.py            # correctness, then benchmark if on TPU
    python colab_svd_mlp.py --bench    # benchmark only
    python colab_svd_mlp.py --shape llama-3b
    python colab_svd_mlp.py --check    # correctness only (runs on CPU too)

On CPU there is no TPU, so Pallas runs in `interpret=True` (the kernel *logic* is
validated, timings are meaningless). On a TPU host the real Mosaic kernels compile.

The math (per MLP weight  W ≈ U @ V,  U:(out,r) V:(r,in), rank r ≪ out,in):

    gate = (x @ V_gate.T) @ U_gate.T ;  up = (x @ V_up.T) @ U_up.T
    h    = silu(gate) * up
    out  = (h @ V_down.T) @ U_down.T

Each projection's rank strip `(BM, r)` is cached in on-chip VMEM and reused across
output-column blocks (the paper's on-chip caching), so it never spills to HBM.
"""

import argparse
import time

import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu


# ===========================================================================
# Hardware spec — auto-detected from the live TPU (no hardcoded v4)
# ===========================================================================
class TPUSpec:
    """Per-core spec used by the block-size model.

    `vmem_bytes` and `ridge` (roofline ridge point, peak-bf16-FLOPs / HBM-bytes/s)
    are the only hardware numbers the model needs. VMEM per generation is only
    approximately documented, so it is treated as a *budget hint*: the runtime
    autotuner (see `svd_swiglu_mlp_auto`) shrinks tiles until the kernel actually
    fits, so an imperfect VMEM guess costs at most one wasted compile, never a
    wrong/over-allocating run.
    """

    def __init__(self, name, vmem_bytes, ridge, tile_m=128, tile_n=512,
                 headroom=0.75):
        self.name = name
        self.vmem_bytes = vmem_bytes
        self.ridge = ridge                 # FLOPs/byte
        self.tile_m = tile_m
        self.tile_n = tile_n
        self.headroom = headroom

    def __repr__(self):
        return (f"TPUSpec({self.name}, VMEM~{self.vmem_bytes/2**20:.0f}MiB, "
                f"ridge~{self.ridge:.0f} FLOPs/byte)")


# Best-effort per-core specs keyed by a substring of `device_kind`.
#   ridge   = peak bf16 FLOPs/s ÷ HBM bandwidth  (Google Cloud TPU specs).
#   vmem    = approximate usable VMEM (conservative; the autotuner corrects it).
# Sources: Google Cloud TPU system-architecture docs; Jouppi et al. (v4,
# arXiv:2304.01433); the JAX "How to Scale Your Model" book.
_MiB = 2 ** 20
_KNOWN_SPECS = [
    # (match substring, name,        vmem,       ridge)
    ("v2",       "TPU v2",   16 * _MiB,  46e12 / 0.7e12),    # ~66
    ("v3",       "TPU v3",   16 * _MiB,  123e12 / 0.9e12),   # ~137
    ("v4",       "TPU v4",   16 * _MiB,  275e12 / 1.2e12),   # ~229
    ("v5 lite",  "TPU v5e",  64 * _MiB,  197e12 / 0.819e12), # ~241   (Colab default)
    ("v5e",      "TPU v5e",  64 * _MiB,  197e12 / 0.819e12),
    ("v6 lite",  "TPU v6e",  128 * _MiB, 918e12 / 1.64e12),  # ~560   (Trillium)
    ("v6e",      "TPU v6e",  128 * _MiB, 918e12 / 1.64e12),
    ("v5",       "TPU v5p",  64 * _MiB,  459e12 / 2.765e12), # ~166   (keep after v5e/v5 lite)
]
# Safe fallback if the kind string is unrecognized: small VMEM forces the
# always-correct K-blocked path; a mid ridge keeps the BM heuristic sane.
_DEFAULT_SPEC = TPUSpec("TPU (unknown)", 16 * _MiB, 220.0)


def detect_spec(verbose=True):
    """Build a `TPUSpec` for the live accelerator (or the safe default on CPU)."""
    if jax.default_backend() != "tpu":
        if verbose:
            print("  [spec] no TPU — using conservative default (interpret mode)")
        return _DEFAULT_SPEC
    kind = jax.devices()[0].device_kind
    low = kind.lower()
    for sub, name, vmem, ridge in _KNOWN_SPECS:
        if sub in low:
            spec = TPUSpec(name, vmem, ridge)
            if verbose:
                print(f"  [spec] device_kind={kind!r} -> {spec}")
            return spec
    if verbose:
        print(f"  [spec] unrecognized device_kind={kind!r} -> {_DEFAULT_SPEC}")
    return _DEFAULT_SPEC


# ===========================================================================
# Analytical block-size model (paper Sec. 4.2.1, Eq. 10 & 11) — first guess
# ===========================================================================
def _ceil_div(a, b):
    return (a + b - 1) // b


def _ceil_mult(x, m):
    return int(_ceil_div(int(x + 0.999999), m) * m)


def _arith_intensity(r, BM, s):
    """Eq. 10: FLOPs per HBM byte for the X·U·V chain."""
    return (2.0 * r) / ((1.0 + r / BM) * s)


def _min_bm_compute_bound(r, s, ridge, tile_m):
    """Smallest BM whose arithmetic intensity reaches the ridge (or None if the
    rank is too small to ever saturate the MXU — a memory-bound signal)."""
    if (2.0 * r) / s <= ridge:
        return None
    bm = r / ((2.0 * r) / (s * ridge) - 1.0)
    return _ceil_mult(bm, tile_m)


def _peak_vmem(BM, BN, K, r, s, kernel, BK):
    """Approx peak VMEM (bytes) for this repo's kernels at a given tiling."""
    if BK is None:                                   # fused: U held whole
        per_set = BM * r * s + K * r * s + 2 * (r * BN * s) + BM * BN * 4
        sets = 2 if kernel == "upgate" else 1
        return BM * K * s + BM * BN * s + sets * per_set
    strip = 2 * (BM * BK * s + BK * r * s) + BM * r * 4 + BM * r * s
    sets = 2 if kernel == "upgate" else 1
    expand = sets * BM * r * s + sets * 2 * (r * BN * s) + BM * BN * s + sets * BM * BN * 4
    return max(strip, expand)


def plan_blocks(M, K, r, N, spec, s, kernel):
    """Return (BM, BN, BK, info) for one projection X(M,K)·U(K,r)·V(r,N).

    Strategy: pick the smallest compute-bound BM, then the widest BN (and, if the
    fused path overflows, the largest BK) that fits the VMEM budget."""
    budget = int(spec.vmem_bytes * spec.headroom)
    bm = _min_bm_compute_bound(r, s, spec.ridge, spec.tile_m)
    mem_bound = bm is None
    if bm is None:
        bm = spec.tile_m
    BM = min(_ceil_mult(bm, spec.tile_m), _ceil_mult(M, spec.tile_m)) if M >= spec.tile_m else M

    def widest_bn(bk):
        best = 0
        cand = spec.tile_n
        while cand <= max(N, spec.tile_n):
            bn = min(cand, N)
            if _peak_vmem(BM, bn, K, r, s, kernel, bk) <= budget:
                best = bn
            else:
                break
            cand += spec.tile_n
        return best

    ai = _arith_intensity(r, BM, s)
    info = dict(ai=ai, compute_bound=ai >= spec.ridge, mem_bound=mem_bound)

    bn = widest_bn(None)                              # 1) prefer fused (U whole)
    if bn > 0:
        info["path"] = "fused"
        return BM, bn, None, info

    bk = (K // spec.tile_m) * spec.tile_m or spec.tile_m   # 2) K-block
    while bk >= spec.tile_m:
        if _peak_vmem(BM, spec.tile_n, K, r, s, kernel, bk) <= budget:
            break
        bk -= spec.tile_m
    bn = widest_bn(bk) or spec.tile_n
    info["path"] = "k-blocked"
    return BM, bn, bk, info


# ===========================================================================
# Pallas plumbing
# ===========================================================================
def _pad_to(x, axis, multiple):
    n = x.shape[axis]
    pad = _ceil_mult(n, multiple) - n
    if pad == 0:
        return x
    widths = [(0, 0)] * x.ndim
    widths[axis] = (0, pad)
    return jnp.pad(x, widths)


# `CompilerParams` (current) vs `TPUCompilerParams` (older JAX) vs neither.
_CP = getattr(pltpu, "CompilerParams", None) or getattr(pltpu, "TPUCompilerParams", None)


def _cp(dimension_semantics):
    """Pipelining hint kwargs, feature-detected so it is a no-op on old/odd JAX."""
    if _CP is None:
        return {}
    try:
        return {"compiler_params": _CP(dimension_semantics=dimension_semantics)}
    except Exception:
        return {}


# --- fused XUV (down projection): O = (X @ U) @ V --------------------------
def _xuv_kernel(x_ref, U_ref, V_ref, o_ref, xu_strip):
    @pl.when(pl.program_id(1) == 0)
    def _():
        xu = jnp.dot(x_ref[...], U_ref[...], preferred_element_type=jnp.float32)
        xu_strip[...] = xu.astype(xu_strip.dtype)          # cache rank strip on-chip
    o = jnp.dot(xu_strip[...], V_ref[...], preferred_element_type=jnp.float32)
    o_ref[...] = o.astype(o_ref.dtype)


def _xuv(X, U, V, BM, BN, interpret):
    M, K = X.shape
    _, r = U.shape
    N = V.shape[1]
    return pl.pallas_call(
        _xuv_kernel,
        grid=(M // BM, N // BN),
        in_specs=[
            pl.BlockSpec((BM, K), lambda m, n: (m, 0)),
            pl.BlockSpec((K, r), lambda m, n: (0, 0)),
            pl.BlockSpec((r, BN), lambda m, n: (0, n)),
        ],
        out_specs=pl.BlockSpec((BM, BN), lambda m, n: (m, n)),
        out_shape=jax.ShapeDtypeStruct((M, N), X.dtype),
        scratch_shapes=[pltpu.VMEM((BM, r), X.dtype)],
        interpret=interpret, name="svd_xuv",
        **_cp(("parallel", "arbitrary")),
    )(X, U, V)


# --- fused Up+Gate: O = silu((X@Ug)@Vg) * ((X@Uu)@Vu) ----------------------
def _upgate_kernel(x_ref, Ug_ref, Vg_ref, Uu_ref, Vu_ref, o_ref, g_strip, u_strip):
    @pl.when(pl.program_id(1) == 0)
    def _():
        g_strip[...] = jnp.dot(x_ref[...], Ug_ref[...],
                               preferred_element_type=jnp.float32).astype(g_strip.dtype)
        u_strip[...] = jnp.dot(x_ref[...], Uu_ref[...],
                               preferred_element_type=jnp.float32).astype(u_strip.dtype)
    gate = jnp.dot(g_strip[...], Vg_ref[...], preferred_element_type=jnp.float32)
    up = jnp.dot(u_strip[...], Vu_ref[...], preferred_element_type=jnp.float32)
    o_ref[...] = (jax.nn.silu(gate) * up).astype(o_ref.dtype)


def _upgate(X, Ug, Vg, Uu, Vu, BM, BN, interpret):
    M, K = X.shape
    _, r = Ug.shape
    N = Vg.shape[1]
    sx = pl.BlockSpec((BM, K), lambda m, n: (m, 0))
    sU = pl.BlockSpec((K, r), lambda m, n: (0, 0))
    sV = pl.BlockSpec((r, BN), lambda m, n: (0, n))
    return pl.pallas_call(
        _upgate_kernel,
        grid=(M // BM, N // BN),
        in_specs=[sx, sU, sV, sU, sV],
        out_specs=pl.BlockSpec((BM, BN), lambda m, n: (m, n)),
        out_shape=jax.ShapeDtypeStruct((M, N), X.dtype),
        scratch_shapes=[pltpu.VMEM((BM, r), X.dtype), pltpu.VMEM((BM, r), X.dtype)],
        interpret=interpret, name="svd_upgate",
        **_cp(("parallel", "arbitrary")),
    )(X, Ug, Vg, Uu, Vu)


# --- K-blocked path: stream U in BK chunks, accumulate the rank strip -------
def _xu_strip_kernel(x_ref, U_ref, o_ref, acc):
    @pl.when(pl.program_id(1) == 0)
    def _():
        acc[...] = jnp.zeros_like(acc)
    acc[...] += jnp.dot(x_ref[...], U_ref[...], preferred_element_type=jnp.float32)
    o_ref[...] = acc[...].astype(o_ref.dtype)


def _xu_strip(X, U, BM, BK, interpret):
    M, K = X.shape
    r = U.shape[1]
    return pl.pallas_call(
        _xu_strip_kernel,
        grid=(M // BM, K // BK),
        in_specs=[
            pl.BlockSpec((BM, BK), lambda m, k: (m, k)),
            pl.BlockSpec((BK, r), lambda m, k: (k, 0)),
        ],
        out_specs=pl.BlockSpec((BM, r), lambda m, k: (m, 0)),
        out_shape=jax.ShapeDtypeStruct((M, r), X.dtype),
        scratch_shapes=[pltpu.VMEM((BM, r), jnp.float32)],
        interpret=interpret, name="svd_xu_strip",
        **_cp(("parallel", "arbitrary")),
    )(X, U)


def _expand_kernel(a_ref, V_ref, o_ref):
    o_ref[...] = jnp.dot(a_ref[...], V_ref[...],
                         preferred_element_type=jnp.float32).astype(o_ref.dtype)


def _expand(A, V, BM, BN, interpret):
    M, r = A.shape
    N = V.shape[1]
    return pl.pallas_call(
        _expand_kernel,
        grid=(M // BM, N // BN),
        in_specs=[pl.BlockSpec((BM, r), lambda m, n: (m, 0)),
                  pl.BlockSpec((r, BN), lambda m, n: (0, n))],
        out_specs=pl.BlockSpec((BM, BN), lambda m, n: (m, n)),
        out_shape=jax.ShapeDtypeStruct((M, N), A.dtype),
        interpret=interpret, name="svd_expand",
        **_cp(("parallel", "arbitrary")),
    )(A, V)


def _expand_swiglu_kernel(g_ref, Vg_ref, u_ref, Vu_ref, o_ref):
    gate = jnp.dot(g_ref[...], Vg_ref[...], preferred_element_type=jnp.float32)
    up = jnp.dot(u_ref[...], Vu_ref[...], preferred_element_type=jnp.float32)
    o_ref[...] = (jax.nn.silu(gate) * up).astype(o_ref.dtype)


def _expand_swiglu(g, Vg, u, Vu, BM, BN, interpret):
    M, r = g.shape
    N = Vg.shape[1]
    sm = pl.BlockSpec((BM, r), lambda m, n: (m, 0))
    sn = pl.BlockSpec((r, BN), lambda m, n: (0, n))
    return pl.pallas_call(
        _expand_swiglu_kernel,
        grid=(M // BM, N // BN),
        in_specs=[sm, sn, sm, sn],
        out_specs=pl.BlockSpec((BM, BN), lambda m, n: (m, n)),
        out_shape=jax.ShapeDtypeStruct((M, N), g.dtype),
        interpret=interpret, name="svd_expand_swiglu",
        **_cp(("parallel", "arbitrary")),
    )(g, Vg, u, Vu)


# ===========================================================================
# Public kernel API
# ===========================================================================
def svd_swiglu_mlp(x, U_gate, V_gate, U_up, V_up, U_down, V_down,
                   *, BM=128, BN=512, BK=None, interpret=False):
    """SVD-compressed SwiGLU MLP (paper Algorithm 2). BK=None holds each U whole in
    VMEM (fastest where it fits); set BK to stream U in contraction chunks."""
    S, H = x.shape
    I, r = U_gate.shape
    H_d, r_d = U_down.shape
    assert V_gate.shape == (r, H) and V_up.shape == (r, H) and U_up.shape == (I, r)
    assert V_down.shape == (r_d, I) and H_d == H

    x_p = _pad_to(x, 0, BM)
    if BK is None:
        h_p = _upgate(x_p, V_gate.T, _pad_to(U_gate.T, 1, BN),
                      V_up.T, _pad_to(U_up.T, 1, BN), BM, BN, interpret)
        out_p = _xuv(h_p, _pad_to(V_down.T, 0, BN), _pad_to(U_down.T, 1, BN),
                     BM, BN, interpret)
        return out_p[:S, :H]

    # K-blocked: up/gate (K=H), then down (K=I)
    xk = _pad_to(x_p, 1, BK)
    g = _xu_strip(xk, _pad_to(V_gate.T, 0, BK), BM, BK, interpret)
    u = _xu_strip(xk, _pad_to(V_up.T, 0, BK), BM, BK, interpret)
    h_p = _expand_swiglu(g, _pad_to(U_gate.T, 1, BN), u, _pad_to(U_up.T, 1, BN),
                         BM, BN, interpret)
    hk = _pad_to(h_p, 1, BK)
    d = _xu_strip(hk, _pad_to(V_down.T, 0, BK), BM, BK, interpret)
    out_p = _expand(d, _pad_to(U_down.T, 1, BN), BM, BN, interpret)
    return out_p[:S, :H]


def _looks_like_vmem_error(e):
    """True only for VMEM-*capacity* errors, which a smaller tile can fix. Hard
    Mosaic compile errors (e.g. 'Bad lhs type' from a precision/dtype mismatch)
    must NOT match — retrying with smaller blocks can't help and only thrashes."""
    s = f"{type(e).__name__}: {e}".lower()
    if "bad lhs type" in s or "bad rhs type" in s:
        return False
    return any(k in s for k in ("vmem", "resource_exhausted", "out of memory",
                                "not enough memory"))


def _ladder(BM, BN, BK0, K_max, try_fused):
    """Configs to try, best→safest. Fused first (if plausibly fits), then ever
    smaller BK / BN. Deduped, order-preserving."""
    bns = [bn for bn in (BN, 512, 256, 128) if bn <= max(BN, 512)]
    bks = [BK0] + [b for b in (2048, 1536, 1024, 512, 256, 128) if b <= K_max]
    if try_fused:
        bks = [None] + bks
    seen, out = set(), []
    for bk in bks:
        for bn in bns:
            key = (BM, bn, bk)
            if key not in seen:
                seen.add(key)
                out.append(key)
    return out


def svd_swiglu_mlp_auto(x, U_gate, V_gate, U_up, V_up, U_down, V_down,
                        *, spec=None, interpret=False, verbose=True):
    """Pick a tiling from the analytical model for the detected TPU, then run —
    automatically falling back to K-blocking / smaller tiles if Mosaic reports a
    VMEM overflow. Returns (output, chosen_config_dict)."""
    if spec is None:
        spec = detect_spec(verbose=verbose)
    S, H = x.shape
    I, r = U_gate.shape
    _, r_d = U_down.shape
    s = jnp.dtype(x.dtype).itemsize

    # Model each projection; take the most conservative (smallest) suggestion.
    bm1, bn1, bk1, i1 = plan_blocks(S, H, r, I, spec, s, "upgate")     # up/gate
    bm2, bn2, bk2, i2 = plan_blocks(S, I, r_d, H, spec, s, "xuv")      # down
    BM = min(bm1, bm2)
    BN = min(bn1, bn2)
    bk_candidates = [b for b in (bk1, bk2) if b]
    BK0 = max(bk_candidates) if bk_candidates else None
    K_max = max(H, I)
    # Only bother trying the fused path first if U-whole is within reach of VMEM.
    try_fused = (max(H, I) * max(r, r_d) * s) <= 3 * spec.vmem_bytes * spec.headroom
    if verbose:
        ip = i1["path"] if i1["path"] == i2["path"] else f"{i1['path']}/{i2['path']}"
        print(f"  [plan] model: BM={BM} BN={BN} BK={BK0} ({ip}); "
              f"AI(up/gate)={i1['ai']:.0f} "
              f"{'compute-bound' if i1['compute_bound'] else 'MEM-bound'}")

    last = None
    for (bm, bn, bk) in _ladder(BM, BN, BK0, K_max, try_fused):
        try:
            out = svd_swiglu_mlp(x, U_gate, V_gate, U_up, V_up, U_down, V_down,
                                 BM=bm, BN=bn, BK=bk, interpret=interpret)
            jax.block_until_ready(out)          # force compile+exec to surface VMEM errors
            cfg = dict(BM=bm, BN=bn, BK=bk,
                       path="fused" if bk is None else "k-blocked")
            if verbose:
                print(f"  [auto] using {cfg}")
            return out, cfg
        except Exception as e:                  # noqa: BLE001 — autotune retry
            if not _looks_like_vmem_error(e):
                raise
            last = e
            if verbose:
                print(f"  [auto] BM={bm} BN={bn} BK={bk} overflowed "
                      f"({type(e).__name__}); shrinking…")
    raise RuntimeError(
        f"No tiling fit VMEM on {spec.name}. Last error: {last}")


# ===========================================================================
# Plain-JAX references (ground truth + dense baseline)
# ===========================================================================
def svd_swiglu_mlp_ref(x, U_gate, V_gate, U_up, V_up, U_down, V_down):
    gate = (x @ V_gate.T) @ U_gate.T
    up = (x @ V_up.T) @ U_up.T
    h = jax.nn.silu(gate) * up
    return (h @ V_down.T) @ U_down.T


def dense_swiglu_mlp_ref(x, W_gate, W_up, W_down):
    gate = x @ W_gate.T
    up = x @ W_up.T
    h = jax.nn.silu(gate) * up
    return h @ W_down.T


# ===========================================================================
# Shapes, inputs, correctness, benchmark
# ===========================================================================
SHAPES = {
    "small":    dict(S=256, H=512,  I=2048,  r=256,  r_d=256),
    "llama-1b": dict(S=128, H=2048, I=8192,  r=1280, r_d=1280),   # Llama-3.2-1B MLP
    "llama-3b": dict(S=128, H=3072, I=8192,  r=1792, r_d=1792),   # Llama-3.2-3B MLP
    "llama-8b": dict(S=128, H=4096, I=14336, r=2560, r_d=2560),   # Llama-3-8B MLP
}

ON_TPU = jax.default_backend() == "tpu"


def _make_factors(S, H, I, r, r_d, dtype, seed=0):
    ks = jax.random.split(jax.random.PRNGKey(seed), 10)
    sc = 1.0 / (H ** 0.5)
    rnd = lambda k, shp, s=1.0: (jax.random.normal(k, shp, jnp.float32) * s).astype(dtype)
    x = rnd(ks[0], (S, H))
    f = (rnd(ks[1], (I, r), sc), rnd(ks[2], (r, H), sc),     # gate U,V
         rnd(ks[3], (I, r), sc), rnd(ks[4], (r, H), sc),     # up   U,V
         rnd(ks[5], (H, r_d), sc), rnd(ks[6], (r_d, I), sc)) # down U,V
    dense = (rnd(ks[7], (I, H), sc), rnd(ks[8], (I, H), sc), rnd(ks[9], (H, I), sc))
    return x, f, dense


def run_correctness(shape="llama-3b"):
    """Assert the Pallas output matches the plain-JAX reference (fp32, tight)."""
    print(f"\n=== correctness ({'real TPU kernels' if ON_TPU else 'CPU interpret'}) ===")
    # fp32 'highest' isolates tiling correctness from MXU rounding for the compare,
    # but it is a GLOBAL flag — restore it in `finally` so a later bf16 benchmark in
    # the same session isn't asked for fp32-precision matmuls on bf16 ('Bad lhs type').
    if ON_TPU:
        jax.config.update("jax_default_matmul_precision", "highest")
    try:
        spec = detect_spec()
        atol = rtol = 2e-2 if ON_TPU else 1e-3
        cases = ["small", shape] if shape != "small" else ["small"]
        for name in cases:
            d = SHAPES[name]
            x, f, _ = _make_factors(**d, dtype=jnp.float32)
            ref = svd_swiglu_mlp_ref(x, *f)
            out, cfg = svd_swiglu_mlp_auto(x, *f, spec=spec, interpret=not ON_TPU,
                                           verbose=False)
            err = float(jnp.max(jnp.abs(out - ref)))
            rel = err / (float(jnp.max(jnp.abs(ref))) + 1e-9)
            ok = bool(jnp.allclose(out, ref, atol=atol, rtol=rtol))
            print(f"  [{name:9s}] {cfg['path']:9s} BN={cfg['BN']} BK={cfg['BK']}  "
                  f"max|err|={err:.2e} rel={rel:.2e} -> {'PASS' if ok else 'FAIL'}")
            assert ok, f"{name}: Pallas diverged from reference"
        print("  all correctness checks passed.")
    finally:
        if ON_TPU:
            jax.config.update("jax_default_matmul_precision", "default")


def _bench(fn, *args, iters=50, warmup=10):
    for _ in range(warmup):
        jax.block_until_ready(fn(*args))
    t0 = time.perf_counter()
    for _ in range(iters):
        out = fn(*args)
    jax.block_until_ready(out)
    return (time.perf_counter() - t0) / iters * 1e3  # ms/iter


def run_benchmark(shape="llama-3b"):
    """Dense vs XLA-fused-SVD vs Pallas-SVD. The Pallas-vs-XLA ratio is the result."""
    d = SHAPES[shape]
    dtype = jnp.bfloat16 if ON_TPU else jnp.float32
    # bf16 wants the fast (default) MXU precision; force it in case a prior
    # correctness run left the global flag on 'highest' (-> 'Bad lhs type' on bf16).
    if ON_TPU:
        jax.config.update("jax_default_matmul_precision", "default")
    spec = detect_spec()
    x, f, (W_gate, W_up, W_down) = _make_factors(**d, dtype=dtype)

    print(f"\n=== benchmark: {shape}  S={d['S']} H={d['H']} I={d['I']} r={d['r']}  "
          f"dtype={dtype.__name__}  backend={jax.default_backend()} ===")

    # Resolve a working Pallas config once (eager autotune), then jit it.
    _, cfg = svd_swiglu_mlp_auto(x, *f, spec=spec, interpret=not ON_TPU)
    dense = jax.jit(lambda x: dense_swiglu_mlp_ref(x, W_gate, W_up, W_down))
    svd_xla = jax.jit(lambda x: svd_swiglu_mlp_ref(x, *f))
    svd_pallas = jax.jit(lambda x: svd_swiglu_mlp(
        x, *f, BM=cfg["BM"], BN=cfg["BN"], BK=cfg["BK"], interpret=not ON_TPU))

    it = 50 if ON_TPU else 3
    t_dense = _bench(dense, x, iters=it)
    t_xla = _bench(svd_xla, x, iters=it)
    t_pallas = _bench(svd_pallas, x, iters=it)
    print(f"  dense MLP (no SVD)   : {t_dense:8.4f} ms/iter")
    print(f"  svd  MLP  (XLA)      : {t_xla:8.4f} ms/iter   ({t_dense/t_xla:5.2f}x vs dense)")
    print(f"  svd  MLP  (Pallas)   : {t_pallas:8.4f} ms/iter   ({t_dense/t_pallas:5.2f}x vs dense)")
    verdict = "Pallas wins" if t_pallas < t_xla else "XLA already wins — kernel not worth it here"
    print(f"  --> Pallas vs XLA    : {t_xla/t_pallas:6.3f}x   ({verdict})")
    if not ON_TPU:
        print("  (CPU/interpret timings are NOT representative — run on a TPU.)")


### 4. Correctness on the real TPU kernels


In [ ]:
run_correctness("llama-3b")


### 5. The benchmark — the number you actually want
`dense` vs XLA's own SVD fusion vs the Pallas kernel. The **Pallas vs XLA** ratio is the result.


In [ ]:
run_benchmark("llama-3b")


### 6. (optional) Sweep all shapes


In [ ]:
for name in ['small', 'llama-1b', 'llama-3b', 'llama-8b']:
    print('===', name, '===')
    run_benchmark(name)


## What to paste back
- `device_kind` (which TPU you got).
- The three timings + the **Pallas vs XLA** ratio.
- Whether each shape ran **fused** or **k-blocked** (the `[auto]` line).
